# NumPy and Numerical Computing: Worked Solutions


In [ ]:
import numpy as np
rng = np.random.default_rng(17)
X = rng.normal(size=(7, 3))
Y = rng.normal(size=(4, 3))


## Problem 1. Centering features
**Problem.** Center each feature of a row matrix without changing the input. Explain the axes and check a singleton batch.

**Solution.** For $X\in\mathbb R^{n\times d}$, $\mu_j=n^{-1}\sum_iX_{ij}$ reduces the observation axis. Retaining shape $(1,d)$ makes $C=X-\mathbf1_n\mu^T$ broadcast across rows. The subtraction allocates a new array. For one row, the centered result is zero.


In [ ]:
def center_features(X):
    X = np.asarray(X, dtype=np.float64)
    if X.ndim != 2 or X.shape[0] == 0:
        raise ValueError("nonempty matrix required")
    return X - X.mean(axis=0, keepdims=True)


In [ ]:
saved = X.copy()
C = center_features(X)
assert C.shape == X.shape
np.testing.assert_allclose(C.mean(axis=0), 0., atol=1e-15)
np.testing.assert_array_equal(X, saved)
np.testing.assert_allclose(center_features(np.array([[3., 5.]])), [[0., 0.]])
# A one-dimensional array has no separate row/column orientation.
assert np.arange(3).T.shape == (3,)


## Problem 2. Pairwise distances
**Problem.** Compute $Q_{ij}=\sum_k(X_{ik}-Y_{jk})^2$ and verify the shape and translation invariance.

**Solution.** The difference tensor has shape $(n,m,d)$; summing its last axis contracts features, leaving the two point indices. This reference implementation stores the entire difference tensor.


In [ ]:
def distances(X, Y):
    X, Y = np.asarray(X), np.asarray(Y)
    if X.ndim != 2 or Y.ndim != 2 or X.shape[1] != Y.shape[1]:
        raise ValueError("matching feature axes required")
    delta = X[:, None, :] - Y[None, :, :]
    return np.sum(delta * delta, axis=-1)


In [ ]:
D = distances(X, Y)
reference = np.array([[sum((a-b)**2) for b in Y] for a in X])
np.testing.assert_allclose(D, reference)
assert D.shape == (7, 4)
np.testing.assert_allclose(distances(X+2, Y+2), D, atol=1e-13)
np.testing.assert_allclose(np.diag(distances(X, X)), 0.)
np.testing.assert_allclose(distances(np.array([[1., 2.]]), np.array([[4., 6.]])), [[25.]])


**Problem.** Compute the same distances with at most $b$ rows at a time; estimate memory and compare with the Gram identity on nearby, large points.

**Solution.** Each block forms $(r,m,d)$ differences for $r\le b$, followed by an $(r,m)$ contraction. Beyond float64 inputs and the $(n,m)$ output, the main temporaries use at most $8(bmd+bm)$ bytes (replace $b$ by $\min(b,n)$). Deleting each difference tensor prevents overlap with the next block allocation. Work remains $O(nmd)$; the output still costs $8nm$ bytes. This accounting excludes array metadata and small library buffers.


In [ ]:
def blocked_distances(X, Y, b):
    X, Y = np.asarray(X, dtype=np.float64), np.asarray(Y, dtype=np.float64)
    if X.ndim != 2 or Y.ndim != 2 or X.shape[1] != Y.shape[1]:
        raise ValueError("matching feature axes required")
    if not isinstance(b, (int, np.integer)) or b < 1:
        raise ValueError("positive integer block size required")
    Q = np.empty((len(X), len(Y)), dtype=np.float64)
    for start in range(0, len(X), b):
        stop = min(start+b, len(X))
        delta = X[start:stop, None, :] - Y[None, :, :]
        Q[start:stop] = np.einsum('ijk,ijk->ij', delta, delta)
        del delta
    return Q


In [ ]:
for n, m, d in [(7, 4, 3), (1, 4, 3), (7, 1, 1)]:
    A, B = rng.normal(size=(n, d)), rng.normal(size=(m, d))
    for block in [1, 3, 20]:
        np.testing.assert_allclose(blocked_distances(A, B, block), distances(A, B))
A = np.array([[1e8+1, 1e8+2]])
B = np.array([[1e8, 1e8]])
direct = blocked_distances(A, B, 1)
gram = np.sum(A*A, axis=1)[:, None] + np.sum(B*B, axis=1)[None, :] - 2*A@B.T
np.testing.assert_array_equal(direct, [[5.]])
print("Direct distance:", direct, "Gram formula:", gram)


The exact squared distance in the last example is $1^2+2^2=5$. The Gram formula subtracts quantities of order $10^{16}$; it gives zero in the tested environment. Direct differences retain the small displacement. A faster matrix multiplication does not guarantee a more accurate small distance.


## Problem 3. Stable logarithmic normalization
**Problem.** Compute the log-sum-exp of every row of a finite matrix, including rows shifted by $1000$.

**Solution.** For each row, let $c=\max_jz_j$. Then $\log\sum_je^{z_j}=c+\log\sum_je^{z_j-c}$. The reduction removes the category axis. All shifted exponents are at most one, avoiding exponential overflow. Small terms can still underflow. The implementation explicitly excludes nonfinite inputs.


In [ ]:
def logsumexp_rows(Z):
    Z = np.asarray(Z, dtype=np.float64)
    if Z.ndim != 2 or Z.shape[1] == 0 or not np.isfinite(Z).all():
        raise ValueError("finite matrix with nonempty rows required")
    c = Z.max(axis=1, keepdims=True)
    return (c + np.log(np.exp(Z-c).sum(axis=1, keepdims=True))).ravel()


In [ ]:
Z = np.array([[0., 1., 2.], [-1., -2., -3.]])
expected = np.log(np.exp(Z).sum(axis=1))
np.testing.assert_allclose(logsumexp_rows(Z), expected)
for shift in [0., 1000., -1000.]:
    np.testing.assert_allclose(logsumexp_rows(Z+shift)-shift, expected, atol=1e-12)
with np.errstate(over="ignore"):
    naive = np.log(np.exp(Z+1000).sum(axis=1))
assert np.isinf(naive).all()
assert np.isfinite(logsumexp_rows(Z+1000)).all()


## Problem 4. Sample covariance
**Problem.** Compute the sample covariance of random rows, check symmetry, and explain positive semidefiniteness.

**Solution.** Set $C=X-\mathbf1_n\bar x^T$ and $S=C^TC/(n-1)$. Then $S^T=S$ and $v^TSv=\|Cv\|^2/(n-1)\ge0$. Both surviving axes index features. Under iid sampling with finite second moments, the denominator $n-1$ makes this an unbiased covariance estimate.


In [ ]:
def sample_covariance(X):
    X = np.asarray(X, dtype=np.float64)
    if X.ndim != 2 or X.shape[0] < 2:
        raise ValueError("at least two observations required")
    C = X - X.mean(axis=0, keepdims=True)
    return C.T @ C / (len(X)-1)


In [ ]:
S = sample_covariance(X)
np.testing.assert_allclose(S, np.cov(X, rowvar=False))
np.testing.assert_allclose(S, S.T)
assert np.linalg.eigvalsh(S).min() >= -1e-12
np.testing.assert_allclose(sample_covariance(X+100), S, atol=1e-12)
C = center_features(X)
for v in rng.normal(size=(10, X.shape[1])):
    np.testing.assert_allclose(v @ S @ v, np.sum((C @ v)**2)/(len(X)-1))
    assert v @ S @ v >= -1e-12


## Problem 5. Three numerical failure mechanisms
**Problem.** Distinguish overflow, cancellation, and input quantization using logarithmic normalization and large-offset variance calculations.

**Solution.** Directly exponentiating the rows shifted by $1000$ overflows. In the variance identity $\overline{x^2}-\bar x^2$, subtracting nearly equal large numbers can erase the variance even when each input is exactly represented. Casting $10^8+(-2,-1,0,1,2)$ to float32 instead loses the deviations *before* the variance is computed. The centered formula cannot recover them. In exact arithmetic both offset examples below have variance $2$.


In [ ]:
for offset in [1e4, 1e8]:
    a = offset + np.arange(-2., 3.)
    for dtype in [np.float64, np.float32]:
        z = a.astype(dtype)
        mu = z.mean()
        naive = np.mean(z*z)-mu*mu
        centered = np.mean((z-mu)**2)
        represented = np.var(z.astype(np.float64))
        print(offset, dtype.__name__, "uncentered:", naive,
              "centered:", centered, "rounded-input variance:", represented)
        expected = 0. if offset == 1e8 and dtype == np.float32 else 2.
        np.testing.assert_allclose(centered, expected)
        np.testing.assert_allclose(represented, expected)


## Problem 6. Views and copies
**Problem.** Compare alternate-column slicing and integer-array row selection. Which operation shares values with the original matrix?

**Solution.** The basic slice is a view with different strides and shared storage. Integer-array indexing produces a copy. The checks test memory sharing directly. A change through the view changes the original; a change through the advanced-indexing copy does not.


In [ ]:
A = np.arange(12.).reshape(3, 4)
view, copy = A[:, ::2], A[[0, 2]]
assert np.shares_memory(A, view)
assert not np.shares_memory(A, copy)
view[0, 0] = -1
assert A[0, 0] == -1
copy[0, 0] = -99
assert A[0, 0] == -1
